# Wedge tuning and vertical resolution

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_wedge_tuning.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 2.3 Tuning and the Wedge
2. 1.5 Bandwidth and Resolution
3. 3.1 Building 2D Earth Models

In [ ]:
# Ask Ogbon starter: Study thin-bed tuning and resolution
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
"""Thin-bed tuning: a wedge of gas sand in shale under a zero-phase Ricker.

Follows Section 2.3 (Tuning and the Wedge) and Section 1.5 (Bandwidth and
Resolution); the Ricker is Section 1.1's.
Units: velocity m/s, thickness m, time ms (two-way), frequency Hz.
Assumptions and limits:
- 1D convolution at normal incidence: the top reflects R and the base -R (the
  same shale above and below); no transmission loss, no multiples, no noise.
- The amplitude is read at the top reflection's time, as Section 2.3 reads it.
- The brightest bed is the Ricker's peak-to-trough time, the square root of 6
  over (2 pi f), not a quarter wavelength; the scan finds it to 0.001 ms.
- The field: the field earth's reservoir, gas sand throughout, wedges out against
  the unconformity; the amplitude at its top is read trace by trace along that
  real pinch-out, where the shale under the sand is not the shale above it.

It prints the numbers Figure 12.5 computes for the same answers as name = value
lines; then it builds the field earth every starter shares, prints what it measures
there as [field] name = value lines, and draws one figure. Python 3.9+, numpy and
matplotlib; the earth's random numbers are seeded, so every run prints the same.
Edit the parameters and run again.
"""
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
LAYER_VP_MPS = 3000.0   # P velocity of the bed (m/s), turns two-way time into thickness
F_HZ = 30.0             # Ricker peak frequency (Hz)
MAX_THICKNESS_M = 40.0  # thickest bed in the wedge (m)

In [ ]:
# --- the book's constants (not interview answers) ---
# the book's standard rocks (SSM.STANDARD, Section 3.3): P velocity (m/s), S velocity (m/s), density (g/cc)
ROCKS = {
    'shale': (2800, 1241.3793103448277, 2.2521031547808517),
    'brine sand': (2889.973997793982, 1504.4719169684909, 2.2545487499999997),
    'oil sand': (2543.1993596875604, 1529.5017037311773, 2.1813626422963988),
    'gas sand': (2341.92404704112, 1582.4208998550723, 2.0379042775100316)
}
N_WEDGE = 80                          # beds in the wedge, evenly spaced in thickness
SWEEP_MAX_MS, SWEEP_N = 60.0, 60001   # the fine scan of two-way bed time: 0 to 60 ms by 0.001 ms
DT_MS, N_FIELD = 1.0, 40              # the field section's sample (ms) and the pinch-out traces read

def report(name, value, fmt='%.6f'):
    """Print one result as name = value, the line Figure 12.5 checks against the book's kernel."""
    print(('%s = ' + fmt) % (name, value))

def style(ax, title, xlabel, ylabel, grid=True):
    """The starters' one look: a title, both axes labelled with their units, a light grid."""
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if grid:
        ax.grid(True, alpha=0.3)

# --- the Ogbon field earth: one seeded 2D earth that every starter shares ---
# In geological order: thin sand and shale beds with a regional dip, folded into an
# asymmetric anticline and cut by normal faults, the largest a listric growth fault
# (its throw grows with depth and its hanging wall rolls over into it); uplift, and an
# angular unconformity that truncates the crest; the reservoir sand subcrops beneath
# it, wedging out against it, shaling out to the left, sealed by a shale cap and
# holding gas above a flat contact; then a draped overburden with a channel. Each
# cell's velocity follows the
# compaction trend at its deepest burial (the rocks under the unconformity were once
# buried deeper), sands soft when shallow and hard when deep; Gardner's density; the
# mudrock and sandstone lines for S. Units: m, m/s, g/cc; times in ms.
FAULTS = [(0.66, 1.0, 1.4, 380.0), (0.80, -1.0, 0.6, 470.0), (0.22, -1.0, 0.5, 560.0)]   # (x / length, side, size, tip m)
COMPACTION = (1900.0, 1.4)                             # P velocity = 1900 m/s + 1.4 per metre of burial, before lithology
K_RES = 9                                              # the reservoir: bed 9 of the deep package, 70 m thick
SEAL_M = 30.0                                          # the shale cap on the reservoir's top: 30 m, and never under two cells

def field_earth(nx=240, nz=320, dx=15.0, dz=5.0, seed=7, crest=0.45, fold=1.0, dip=1.0, faults=None, throw=1.0,
                unconformity=True, seal=None, upper=None, lower=None, column_m=130.0):
    """The field earth on nz by nx cells of dz by dx (m). seal, upper and lower are (vp, vs, density) of the shale on the
    unconformity and of the reservoir above and below its flat contact, or None for the compaction trend's own rocks."""
    rng = np.random.default_rng(seed)
    x, z = np.arange(nx) * dx, np.arange(nz) * dz
    xn, zz, sc = x / x[-1], z[:, None] + 0.0 * x, nz * dz / 1600.0     # sc: the geometry scales with the grid's depth
    hump = np.exp(-0.5 * ((xn - crest) / np.where(xn < crest, 0.13, 0.25)) ** 2)   # the anticline, steep to the left
    swell = 1.0 + 0.15 * np.sin(2 * np.pi * (0.8 * xn + 0.37 * np.arange(26)[:, None]))   # each bed thickens and thins
    thick = sc * (24.0 + 22.0 * rng.random(26))[:, None] * swell
    thick[K_RES] = sc * 70.0
    beds = sc * 470.0 + np.cumsum(thick, axis=0) - thick + sc * (170.0 * dip * xn - 300.0 * fold * hump)
    if faults is not None:
        fs = [(rng.uniform(0.12, 0.88), rng.choice([-1.0, 1.0]), rng.uniform(0.4, 1.4), rng.uniform(300.0, 600.0))
              for k in range(faults)]
    fault = np.zeros((nz, nx), bool)
    for x0, side, size, tip in (FAULTS if faults is None else fs):
        tip, a = tip * sc, 0.2 / (1200.0 * sc) ** 1.3       # the plane: x / length = x0 + side a (depth - tip) to the 1.3, listric
        plane = np.where(side * (xn - x0) > 0, tip + (np.abs(xn - x0) / a) ** (1 / 1.3), -np.inf)   # its depth at each x
        for k in range(26):
            g = throw * size * sc * 90.0 * np.clip((beds[k] - tip) / (700.0 * sc), 0.0, 1.6)      # growth: deeper beds slip more
            xk = x0 + side * a * np.maximum(beds[k] - tip, 0.0) ** 1.3                          # where the plane cuts the bed
            g = g * (1.0 + 0.6 * np.exp(-np.abs(xn - xk) / 0.04))                                # the rollover into the fault
            beds[k] = beds[k] + np.clip(plane - beds[k], 0.0, g)       # the hanging wall drops; in the heave the bed lies on the plane
        ix = np.rint((x0 + side * a * np.maximum(z - tip, 0.0) ** 1.3) * (nx - 1)).astype(int)
        on = (z > tip) & (ix >= 0) & (ix < nx)
        fault[np.nonzero(on)[0], ix[on]] = True
    beds = np.maximum.accumulate(beds, axis=0)
    if unconformity:                                   # cut just under the reservoir's crest, so it subcrops on both flanks
        unc = beds[K_RES + 1, np.argmax(hump)] + sc * (15.0 + 160.0 * dip * (xn - crest)
                                                       + 10.0 * np.sin(2 * np.pi * (1.3 * xn + 0.2)))
    else:
        unc = beds[0] - sc * 30.0                      # no erosion: the overburden lies on the beds
    fault &= zz > unc                                  # the faults are older than the unconformity
    fr = np.cumsum(0.6 + rng.random(13))
    fr = (fr - fr[0]) / (fr[-1] - fr[0])
    over = [sc * 40.0 + (unc.min() - sc * 90.0) * f + 0.85 * f * f * (unc - unc.min()) - sc * 25.0 * f * hump for f in fr]
    tops = np.maximum.accumulate(np.vstack(over + list(np.maximum(beds, unc))), axis=0)
    lay = (zz[:, None, :] >= tops[None, :, :]).sum(axis=1)                    # layer 13 is the seal; bed k is layer 14 + k
    sand = np.where(lay < 13, rng.random(41)[lay] < 0.45, (lay - 14) % 2 == 1) & (lay > 0) & (lay != 13)
    chan = (zz >= over[5]) & (zz < over[5] + sc * 45.0 * np.exp(-0.5 * ((xn - 0.7) / 0.04) ** 2))
    res = lay == 14 + K_RES
    top = np.where(res, zz, np.inf).min(axis=0)
    cap = (zz < top) & (zz >= top - max(SEAL_M * sc, 2.0 * dz))   # the seal: shale on the sand's top, under a bed or the unconformity
    sand = np.where(cap, False, sand)
    burial = zz + np.where(lay >= 14, sc * 150.0 * hump, 0.0)                # uplift: the deep rocks were buried deeper
    vary = 1.0 + 0.035 * np.clip(rng.standard_normal(41), -1.2, 1.2)        # bed to bed variety
    vp = (COMPACTION[0] + COMPACTION[1] * burial) * vary[lay]
    vp = np.where(sand | chan, vp * (0.86 + 0.1 * np.minimum(burial / 1000.0, 1.8)), vp)
    vp = np.where(chan, vp * 0.92, vp)                 # the channel cuts a horizon and is filled with a soft sand
    sand |= chan
    vs = np.where(sand, 0.804 * vp - 856.0, (vp - 1360.0) / 1.16)            # the sandstone and mudrock lines (Castagna)
    den = 1.741 * (vp / 1000.0) ** 0.25                                      # Gardner
    ntg = np.clip((xn - 0.12) / 0.12, 0.0, 1.0)                              # the reservoir shales out to the left
    contact = float(top.min()) + column_m * sc         # the flat contact, column_m under the shallowest sand
    for rock, on in ((seal, (lay == 13) | cap), (upper, res & (zz < contact)), (lower, res & (zz >= contact))):
        if rock is not None:
            w = 1.0 if rock is seal else ntg
            vp, vs, den = [np.where(on, w * rock[i] + (1 - w) * q, q) for i, q in enumerate((vp, vs, den))]
    return dict(x=x, z=z, vp=vp, vs=vs, den=den, lay=lay, sand=sand, res=res, fault=fault, unc=unc, top=top,
                contact=contact, ntg=ntg, dx=dx, dz=dz, chan=chan, cap=cap, gas=res & (zz < contact) & (ntg > 0.5))

def field_time(m, dt_ms, t_max_ms):
    """Each trace from depth to two-way time by vertical rays: every cell array of m on a dt_ms grid to t_max_ms."""
    t_top = 2000.0 * np.cumsum(m['dz'] / m['vp'], axis=0) - 2000.0 * m['dz'] / m['vp']
    t = np.arange(0.0, t_max_ms, dt_ms)
    pick = np.array([np.searchsorted(t_top[:, i], t, 'right') - 1 for i in range(t_top.shape[1])]).T
    cols = np.arange(t_top.shape[1])[None, :]
    out = dict((k, m[k][pick, cols]) for k in ('vp', 'vs', 'den', 'lay', 'sand', 'res', 'gas'))
    seen = np.vstack([0 * cols, np.cumsum(m['fault'], axis=0)])                 # fault cells above each row
    after = np.maximum(np.vstack([pick[1:], pick[-1:] + 1]), pick + 1)
    out['fault'] = seen[np.minimum(after, seen.shape[0] - 1), cols] > seen[pick, cols]   # a fault cell inside the sample
    out['t'], out['t_top'] = t, t_top
    return out

def reflectivity(ai):
    """Normal-incidence coefficient at the top of every sample, down axis 0 (Section 2.1)."""
    rc = np.zeros_like(ai)
    rc[1:] = (ai[1:] - ai[:-1]) / (ai[1:] + ai[:-1])
    return rc

def q_peak_hz(f0, t_s, q):
    """Peak (Hz) of a Ricker of peak f0 after t_s seconds through constant Q: the root of 2 f f / (f0 f0) + (pi t / Q) f = 2."""
    b = np.pi * t_s / q
    return (-b + np.sqrt(b * b + 16.0 / f0 ** 2)) * f0 ** 2 / 4.0

def field_image(rc, dt_ms, f0, q=80.0, snr=None, seed=11):
    """Every trace convolved with a zero-phase Ricker whose peak falls with time as constant q says, plus coloured noise."""
    nt, dt = rc.shape[0], dt_ms * 1e-3
    f = np.fft.rfftfreq(2 * nt, dt)[:, None]
    def band(fp):                                      # the Ricker's spectrum (Section 1.1), over dt for a sum of samples
        return 2.0 / np.sqrt(np.pi) * f ** 2 / fp ** 3 * np.exp(-(f / fp) ** 2) / dt
    def filt(a, fp):
        return np.fft.irfft(np.fft.rfft(a, 2 * nt, axis=0) * band(fp), 2 * nt, axis=0)[:nt]
    fp = q_peak_hz(f0, np.arange(nt) * dt, q)
    knots = np.linspace(fp.min(), fp.max(), 4 if fp.max() - fp.min() > 1e-9 else 1)
    step = max((fp.max() - fp.min()) / 3.0, 1e-9)
    out = sum(np.clip(1.0 - np.abs(fp - k) / step, 0.0, 1.0)[:, None] * filt(rc, k) for k in knots)
    if snr:
        noise = filt(np.random.default_rng(seed).standard_normal(rc.shape), fp.mean())
        noise = (noise + np.roll(noise, 1, 1) + np.roll(noise, -1, 1)) / 3.0          # coherent over a few traces
        out = out + noise * np.sqrt((out ** 2).mean() / (noise ** 2).mean()) / snr
    return out

def fieldnote(name, value, fmt='%.3f'):
    """Print a number of the field earth as [field] name = value: computed here, not by Figure 12.5."""
    print(('[field] %s = ' + fmt) % (name, value))

def show(ax, a, x_m, y, title, ylabel='two-way time (ms)', clip=None):
    """A section as an image, black for positive, clipped at clip (or at its 99th percentile): km across, time or depth down."""
    c = clip if clip else np.percentile(np.abs(a), 99.0) + 1e-12
    ax.imshow(a, cmap='gray_r', vmin=-c, vmax=c, aspect='auto', extent=[x_m[0] / 1e3, x_m[-1] / 1e3, y[-1], y[0]])
    style(ax, title, 'distance (km)', ylabel, grid=False)

def ricker_at(t_ms, f_hz):
    """The zero-phase Ricker of peak f_hz (Hz) at time t_ms (ms) from its peak (Section 1.1)."""
    a = (np.pi * f_hz * t_ms * 1e-3) ** 2
    return (1 - 2 * a) * np.exp(-a)

def bed_coefficient(rocks):
    """Coefficient of the top of a gas sand bed in shale; the base reflects its negative."""
    z_shale = rocks['shale'][0] * rocks['shale'][2]
    z_gas = rocks['gas sand'][0] * rocks['gas sand'][2]
    return (z_gas - z_shale) / (z_gas + z_shale)

def top_amplitude(tau_ms, r, f_hz):
    """The trace at the top of a bed of two-way time tau_ms: R w(0) - R w(bed_ms) (Section 2.3)."""
    return r * (ricker_at(0.0, f_hz) - ricker_at(tau_ms, f_hz))

def validate(vp_mps, f_hz, max_thickness_m):
    """Stop with a clear message on a parameter the model cannot use."""
    if vp_mps <= 0 or f_hz <= 0 or max_thickness_m <= 0:
        raise ValueError('LAYER_VP_MPS, F_HZ and MAX_THICKNESS_M must all be positive')
    if 1000.0 * np.sqrt(6.0) / (2 * np.pi * f_hz) >= SWEEP_MAX_MS:
        raise ValueError('at %.1f Hz the tuning time is past the %.0f ms scan: raise SWEEP_MAX_MS' % (f_hz, SWEEP_MAX_MS))

def pinch_out(e, t, img, n):
    """The n traces downdip of where the reservoir wedges out on the left flank: x (m), two-way bed time (ms),
    the amplitude at its top, the top's time (ms), and the earth's own coefficients at the top and the base of the
    thickest of them: the cap over the sand, and the sand over the bed beneath it."""
    thick = e['res'].sum(axis=0) * e['dz']
    ix = np.arange(np.nonzero(thick == 0)[0].min() - 1, -1, -1)[:n]   # leftward from the crest, where the sand is cut away
    bed_ms = 2000.0 * thick[ix] / ROCKS['gas sand'][0]           # its time thickness in the gas sand
    k = np.argmax(t['res'][:, ix], axis=0)                       # the sample at the top of the sand
    c = ix[np.argmax(thick[ix])]
    rows = np.nonzero(e['res'][:, c])[0]
    z = e['vp'][:, c] * e['den'][:, c]
    r_top, r_base = [(z[i] - z[i - 1]) / (z[i] + z[i - 1]) for i in (rows[0], rows[-1] + 1)]
    return e['x'][ix], bed_ms, img[k, ix], t['t'][k], r_top, r_base

def plot_tuning(e, t, img, xs, ts, bed_ms, amp, fine_ms, fine_amp, own_amp, r_top, r_base, tune_ms):
    """The pinch-out in the field section, and its amplitude against the bed's time thickness, with the 1D wedge's curve."""
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    show(ax[0], img, e['x'], t['t'], 'The reservoir wedges out against the unconformity')
    ax[0].plot(xs / 1e3, ts, 'r.', ms=3, label='top of the sand, %d traces' % xs.size)
    ax[0].set_xlim(xs[-1] / 1e3 - 0.3, xs[0] / 1e3 + 0.6)
    ax[0].set_ylim(ts.max() + 80, ts.min() - 80)
    ax[0].legend(fontsize=8, loc='lower left')
    ax[1].plot(fine_ms, fine_amp, 'k', lw=1, label='the 1D wedge, Section 2.3')
    ax[1].plot(fine_ms, own_amp, 'k--', lw=1, label='the same, with the field\'s own top %+.3f and base %+.3f' % (r_top, r_base))
    ax[1].plot(bed_ms, np.abs(amp), 'o', ms=4, color='tab:red', label='the field pinch-out, trace by trace; the cap\'s top, %.0f m up, adds the rest' % SEAL_M)
    ax[1].axvline(tune_ms, color='r', ls='--', label='tuning, %.1f ms' % tune_ms)
    ax[1].axvline(500.0 / F_HZ, color='0.5', ls=':', label='quarter wavelength, %.1f ms' % (500.0 / F_HZ))
    style(ax[1], 'Tuning along the pinch-out at %.0f Hz' % F_HZ, 'bed thickness, two-way time (ms)',
          'amplitude at the top (coefficient units)')
    ax[1].set_xlim(0, max(bed_ms.max(), 2.2 * tune_ms))
    ax[1].set_ylim(0, 1.45 * max(own_amp.max(), np.abs(amp).max()))
    ax[1].legend(fontsize=7, loc='upper right')
    fig.tight_layout()
    plt.show()

def main():
    validate(LAYER_VP_MPS, F_HZ, MAX_THICKNESS_M)
    r = bed_coefficient(ROCKS)
    h_m = np.linspace(MAX_THICKNESS_M / N_WEDGE, MAX_THICKNESS_M, N_WEDGE)
    tau_ms = 2.0 * h_m / LAYER_VP_MPS * 1000.0             # two-way time through each bed
    fine_ms = np.linspace(0.0, SWEEP_MAX_MS, SWEEP_N)
    fine_amp = np.abs(top_amplitude(fine_ms, r, F_HZ))
    tune_ms = fine_ms[fine_amp.argmax()]
    tune_m = tune_ms * LAYER_VP_MPS / 2000.0                # half the two-way time (s) times the velocity
    quarter_m = LAYER_VP_MPS / (4.0 * F_HZ)
    report('tuning_ms', tune_ms, '%.2f')
    report('tuning_m', tune_m, '%.2f')
    report('quarter_wavelength_m', quarter_m, '%.2f')
    report('peak_over_R', fine_amp.max() / abs(r), '%.4f')
    report('tuning_in_sweep', tau_ms.min() <= tune_ms <= tau_ms.max(), '%d')
    # self-checks: the scan finds the closed forms of Section 2.3
    assert abs(tune_ms - 1000.0 * np.sqrt(6.0) / (2 * np.pi * F_HZ)) < 2e-3
    assert abs(fine_amp.max() / abs(r) - (1 + 2 * np.exp(-1.5))) < 1e-6
    # the field: gas sand in the whole reservoir, under the standard shale, one wavelet, no noise
    e = field_earth(nz=640, dz=2.5, seal=ROCKS['shale'], upper=ROCKS['gas sand'], lower=ROCKS['gas sand'])
    t = field_time(e, DT_MS, 1000.0)
    img = field_image(reflectivity(t['vp'] * t['den']), DT_MS, F_HZ, q=np.inf)
    xs, bed_ms, amp, ts, r_top, r_base = pinch_out(e, t, img, N_FIELD)
    # the shale under the sand is the compaction trend's, not the cap's: its base reflects r_base, not -r_top
    own_amp = np.abs(r_top * ricker_at(0.0, F_HZ) + r_base * ricker_at(fine_ms, F_HZ))
    best = int(np.argmax(np.abs(amp)))
    fieldnote('pinchout_traces', xs.size, '%d')
    fieldnote('pinchout_thinnest_ms', bed_ms.min(), '%.2f')
    fieldnote('pinchout_brightest_ms', bed_ms[best], '%.2f')
    fieldnote('pinchout_brightest_m', bed_ms[best] * ROCKS['gas sand'][0] / 2000.0, '%.2f')
    fieldnote('gas_sand_tuning_m', tune_ms * ROCKS['gas sand'][0] / 2000.0, '%.2f')
    fieldnote('top_r', r_top, '%.4f')
    fieldnote('base_r', r_base, '%.4f')
    fieldnote('brightest_amplitude', abs(amp[best]), '%.4f')
    fieldnote('own_top_and_base_peak', own_amp.max(), '%.4f')
    fieldnote('own_top_and_base_tuning_ms', fine_ms[own_amp.argmax()], '%.2f')
    fieldnote('off_own_curve_max', np.abs(np.abs(amp) - np.interp(bed_ms, fine_ms, own_amp)).max(), '%.4f')
    plot_tuning(e, t, img, xs, ts, bed_ms, amp, fine_ms, fine_amp, own_amp, r_top, r_base, tune_ms)

if __name__ == '__main__':
    main()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)